# CPP Latent-Dynamics Pipeline — Self-Contained Notebook

**Every class and function is defined directly in the cells below.**  
No external `.py` files need to be imported.  Run cells **top → bottom** to reproduce the full analysis.

| # | Stage | Key output |
|---|-------|-----------|
| 0 | Imports & path setup | — |
| 1 | Configuration classes | — |
| 2 | Utility functions | — |
| 3 | Data contract validator | `Results/validation/` |
| 4 | Dataset pipeline | — |
| 5 | Model: CPPForwardGRU + loss | — |
| 6 | Training pipeline | — |
| **▶ 7** | **Run training** | `Results/model_checkpoints/best_model.pt` |
| **▶ 8** | **Export latent states** | `Data/IntermediateData/latents_full/latents_full.npz` |
| 9 | Ridge regression (RT analysis) | — |
| **▶ 10** | **Run regression** | `Results/regression/` |
| 11 | Results & next steps | — |


## 0 · Library Imports & Path Setup

All third-party libraries used anywhere in the pipeline are imported here in one place.  
**Always run this cell first.**


In [1]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Tuple
import json
import random
from typing import Any, Dict
import numpy as np
import torch
from dataclasses import asdict
from typing import Dict, List, Tuple
import pandas as pd
from dataclasses import dataclass
from torch.utils.data import DataLoader, Dataset
from typing import Dict, Tuple
import torch.nn as nn
import shutil
import time
from dataclasses import replace
from torch.utils.data import DataLoader
from s1_modeling.config import TrainingConfig
from s1_modeling.dataset import load_stage2_dataset, make_dataloaders
from s1_modeling.model import CPPForwardGRU, ForwardOutputs, masked_self_supervised_loss
from s1_modeling.utils import set_global_seed
import matplotlib.pyplot as plt
import sys
import csv
from typing import Any, Dict, List, Optional, Sequence, Tuple
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from IPython.display import Image, display
# ── Locate project root ───────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "conftest.py").exists():
        PROJECT_ROOT = _c
        break

# ── Canonical paths ───────────────────────────────────────────────────────────
DATASET_DIR      = PROJECT_ROOT / "Data" / "ProcessedData"
INTERMEDIATE_DIR = PROJECT_ROOT / "Data" / "IntermediateData" / "latents_full"
RESULTS_DIR      = PROJECT_ROOT / "Results"
CHECKPOINT_PATH  = RESULTS_DIR  / "model_checkpoints" / "best_model.pt"
LATENT_PATH      = INTERMEDIATE_DIR / "latents_full.npz"

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}  (exists={DATASET_DIR.exists()})")
print(f"Checkpoint   : {CHECKPOINT_PATH}  (exists={CHECKPOINT_PATH.exists()})")
print(f"Latents      : {LATENT_PATH}  (exists={LATENT_PATH.exists()})")

Project root : /Users/siyu/Documents/GitHub/biiigProject
Dataset dir  : /Users/siyu/Documents/GitHub/biiigProject/Data/ProcessedData  (exists=True)
Checkpoint   : /Users/siyu/Documents/GitHub/biiigProject/Results/model_checkpoints/best_model.pt  (exists=True)
Latents      : /Users/siyu/Documents/GitHub/biiigProject/Data/IntermediateData/latents_full/latents_full.npz  (exists=False)


### Quick dataset preview

Confirm the processed dataset is present and shaped as expected.

In [2]:
_eeg   = np.load(DATASET_DIR / "eeg_cpp_trials.npy")
_times = np.load(DATASET_DIR / "times_ms.npy")
_meta  = pd.read_csv(DATASET_DIR / "metadata.csv")

print(f"EEG shape : {_eeg.shape}  (trials × timepoints × channels)")
print(f"Time axis : {_times[0]:.0f} ms → {_times[-1]:.0f} ms  ({len(_times)} points)")
print(f"Metadata  : {len(_meta)} rows × {len(_meta.columns)} columns")
print(f"Columns   : {list(_meta.columns)}")

EEG shape : (7297, 308, 3)  (trials × timepoints × channels)
Time axis : -1000 ms → 200 ms  (308 points)
Metadata  : 7297 rows × 28 columns
Columns   : ['original_row_index', 'probe_accuracy', 'probe_rt', 'probe_attribute', 'cue_dimensionality', 'probe_leftrightwin', 'cue_color', 'cue_direction', 'cue_size', 'cue_luminance', 'rt_is_not_outlier', 'subj_idx', 'stim_onset', 'stim_code', 'resp_onset_sample', 'subj_id', 'subject_id', 'within_subject_trial_index', 'trial_id', 'RT_ms', 'correctness', 'condition', 'difficulty', 'evidence_strength', 'choice', 'response_hand', 'artifact_rejection_flag', 'alignment']


## 1 · Configuration Classes

Three **frozen dataclasses** that carry every hyperparameter in the pipeline.  
No raw numbers should appear anywhere else in the code — change a parameter here and it propagates everywhere.

| Class | What it controls |
|-------|-----------------|
| `DataContractConfig` | Required files, expected channels, required metadata columns |
| `ModelConfig` | Encoder architecture: `projection_dim`, `hidden_dim`, `num_layers` |
| `LossWeights` | All 11 `lambda_*` weights · shape-prior flags · loss-related time windows |
| `TrainingConfig` | Training loop + nested `model: ModelConfig` + `loss: LossWeights`; `@property` shims keep old `config.hidden_dim` call-sites working |
| `AnalysisConfig` | PCA components, RT-bin quantiles, post-training time windows |

#### `config.py` — § 1  Data Contract Configuration

In [3]:
@dataclass(frozen=True)
class DataContractConfig:
    """Stage 1 contract: expected files, channels, and required metadata columns.

    Encodes the structural requirements that the processed EEG dataset directory
    must satisfy before any model training can begin.
    """

    expected_files: Tuple[str, ...] = (
        "eeg_cpp_trials.npy",
        "metadata.csv",
        "times_ms.npy",
        "channel_names.txt",
        "preprocessing_notes.md",
    )
    expected_channel_order: Tuple[str, ...] = ("CP1", "CP2", "CPz")
    required_metadata_columns: Tuple[str, ...] = (
        "trial_id",
        "alignment",
    )
    optional_aliases: dict = field(default_factory=dict)

#### `config.py` — § 2  Model Architecture Configuration

In [4]:
@dataclass(frozen=True)
class ModelConfig:
    """Architectural hyperparameters for CPPForwardGRU.

    Separating architecture from training and loss concerns makes it easy to
    swap encoder depth or projection size without touching loss weights.
    """

    projection_dim: int = 16
    """Dimensionality of the linear input-projection layer (before LayerNorm)."""

    hidden_dim: int = 32
    """GRU hidden-state dimensionality; also the latent-space dimensionality."""

    num_layers: int = 1
    """Number of stacked GRU layers (set > 1 for deeper recurrent encoding)."""

#### `config.py` — § 3  Loss Weights & Shape-Prior Configuration

In [5]:
@dataclass(frozen=True)
class LossWeights:
    """Weights and windows that control the self-supervised composite loss.

    Each ``lambda_*`` field scales one loss term.  Setting a weight to 0.0
    effectively disables that term without touching the model or training loop.
    The CPP shape-prior group (monotonic / slope_floor / late_amplitude /
    cpp_mean_alignment) can be disabled wholesale via ``enable_cpp_shape_prior``.

    Loss term overview
    ------------------
    lambda_recon             : MSE between reconstructed and real EEG (per-channel).
    lambda_future            : MSE between predicted and real future EEG.
    lambda_derivative        : MSE between reconstructed and real first-order slope.
    lambda_variance          : Channel-level variance alignment between recon and real.
    lambda_cpp_mean          : MSE on the 3-channel CPP proxy (mean across channels).
    lambda_cpp_prior         : Global scale for the CPP shape-prior sub-group.
    lambda_monotonic         : Penalises downward steps in the reconstructed CPP.
    lambda_slope_floor       : Penalises recon slope falling below a fraction of target.
    lambda_late_amplitude    : Penalises under-shooting the late CPP amplitude.
    lambda_cpp_mean_alignment: Alignment loss on CPP proxy over the analysis window.
    lambda_smooth            : Penalises large frame-to-frame jumps in latent state.
    """

    lambda_recon: float = 1.0
    lambda_future: float = 0.2
    lambda_derivative: float = 0.5
    lambda_variance: float = 0.5
    lambda_cpp_mean: float = 0.5
    lambda_cpp_prior: float = 0.1
    lambda_monotonic: float = 1.0
    lambda_slope_floor: float = 0.5
    lambda_late_amplitude: float = 1.0
    lambda_cpp_mean_alignment: float = 0.05
    lambda_smooth: float = 0.001

    future_weight_scale: float = 0.75
    """Scaling factor applied to future-prediction mask weights."""

    slope_floor_ratio: float = 0.5
    """Fraction of the target slope used as the floor for slope_floor_loss."""

    enable_cpp_shape_prior: bool = True
    """Toggle the entire CPP shape-prior sub-group on or off."""

    # Time windows (ms, response-locked) used inside the loss computation.
    analysis_window_ms: Tuple[float, float] = (-600.0, -50.0)
    late_window_ms: Tuple[float, float] = (-120.0, -50.0)

#### `config.py` — § 4  Training Loop & Data-Pipeline Configuration

In [6]:
@dataclass(frozen=True)
class TrainingConfig:
    """Training loop, data-pipeline, and split configuration.

    Architecture hyperparameters live in the nested ``model`` field
    (a :class:`ModelConfig` instance), and all loss weights live in the
    nested ``loss`` field (a :class:`LossWeights` instance).

    Example
    -------
    >>> cfg = TrainingConfig(
    ...     max_epochs=50,
    ...     model=ModelConfig(hidden_dim=64),
    ...     loss=LossWeights(lambda_smooth=0.01, enable_cpp_shape_prior=False),
    ... )
    """

    # --- Reproducibility --------------------------------------------------
    seed: int = 42

    # --- Data pipeline ----------------------------------------------------
    batch_size: int = 64
    train_fraction: float = 0.70
    val_fraction: float = 0.15
    test_fraction: float = 0.15
    future_horizon_ms: int = 50
    """Length of the causal prediction horizon in milliseconds."""

    # --- Optimiser --------------------------------------------------------
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    gradient_clip: float = 1.0

    # --- Training schedule ------------------------------------------------
    max_epochs: int = 100
    early_stopping_patience: int = 15

    # --- Time-window helpers (used by dataset weighting and analysis) ------
    analysis_window_ms: Tuple[float, float] = (-600.0, -50.0)
    early_window_ms: Tuple[float, float] = (-600.0, -300.0)
    mid_window_ms: Tuple[float, float] = (-300.0, -120.0)
    late_window_ms: Tuple[float, float] = (-120.0, -50.0)

    # --- Nested sub-configurations ----------------------------------------
    model: ModelConfig = field(default_factory=ModelConfig)
    """Architecture hyperparameters (projection_dim, hidden_dim, num_layers)."""

    loss: LossWeights = field(default_factory=LossWeights)
    """All loss weights, shape-prior flags, and loss-related time windows."""

    # ------------------------------------------------------------------
    # Backwards-compatibility shims
    # ------------------------------------------------------------------
    # The properties below expose the most-used sub-fields directly on
    # TrainingConfig so that existing call sites (model instantiation,
    # sweep parameter replacement) continue to work without modification.
    # New code should prefer ``config.model.*`` and ``config.loss.*``.

    @property
    def projection_dim(self) -> int:
        """Shortcut → config.model.projection_dim."""
        return self.model.projection_dim

    @property
    def hidden_dim(self) -> int:
        """Shortcut → config.model.hidden_dim."""
        return self.model.hidden_dim

    @property
    def num_layers(self) -> int:
        """Shortcut → config.model.num_layers."""
        return self.model.num_layers

    @property
    def lambda_recon(self) -> float:
        return self.loss.lambda_recon

    @property
    def lambda_future(self) -> float:
        return self.loss.lambda_future

    @property
    def lambda_derivative(self) -> float:
        return self.loss.lambda_derivative

    @property
    def lambda_variance(self) -> float:
        return self.loss.lambda_variance

    @property
    def lambda_cpp_mean(self) -> float:
        return self.loss.lambda_cpp_mean

    @property
    def lambda_cpp_prior(self) -> float:
        return self.loss.lambda_cpp_prior

    @property
    def lambda_monotonic(self) -> float:
        return self.loss.lambda_monotonic

    @property
    def lambda_slope_floor(self) -> float:
        return self.loss.lambda_slope_floor

    @property
    def lambda_late_amplitude(self) -> float:
        return self.loss.lambda_late_amplitude

    @property
    def lambda_cpp_mean_alignment(self) -> float:
        return self.loss.lambda_cpp_mean_alignment

    @property
    def lambda_smooth(self) -> float:
        return self.loss.lambda_smooth

    @property
    def future_weight_scale(self) -> float:
        return self.loss.future_weight_scale

    @property
    def slope_floor_ratio(self) -> float:
        return self.loss.slope_floor_ratio

    @property
    def enable_cpp_shape_prior(self) -> bool:
        return self.loss.enable_cpp_shape_prior

#### `config.py` — § 5  Analysis / Readout Configuration

In [7]:
@dataclass(frozen=True)
class AnalysisConfig:
    """Stage 3 & 4 latent-readout and decoding settings.

    Governs PCA decomposition, RT-bin boundaries, and the time windows used
    for latent-state analyses after training.
    """

    response_locked_window_ms: Tuple[int, int] = (-600, -50)
    """Analysis window used for response-locked latent extraction."""

    contaminated_window_ms: Tuple[int, int] = (-50, 100)
    """Post-response window excluded from causal analyses."""

    pca_components: int = 3
    """Number of principal components retained in latent PCA."""

    rt_bin_quantiles: Tuple[float, float] = (0.33, 0.66)
    """Quantile boundaries for fast / medium / slow RT tertile binning."""

    evidence_bin_quantiles: Tuple[float, ...] = (0.50,)
    """Quantile boundaries for low / high evidence median split."""

#### `config.py` — § 6  Path Helpers

In [8]:
def default_evidence_dir(root: Path) -> Path:
    """Return the canonical evidence output directory for a project root."""
    return root / "Results" / "stage2_modeling"

## 2 · Utility Functions

`set_global_seed(seed)` — seeds Python, NumPy, and PyTorch simultaneously for full reproducibility.  
`ensure_dir(path)` — creates a directory and all parents if they do not exist.  
`save_json(data, path)` — serialises to JSON, creating parent dirs automatically.  
`safe_float(x)` — converts to float, returning `NaN` on failure.

In [9]:
def set_seed(seed: int) -> None:
    """Set all global random seeds for reproducibility.

    Covers Python ``random``, NumPy, and PyTorch (CPU + CUDA).

    Parameters
    ----------
    seed : Integer seed value.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Backwards-compatibility alias (old code used set_global_seed).
set_global_seed = set_seed


def ensure_dir(path: Path) -> Path:
    """Create *path* and all parents if they do not exist, then return *path*.

    Parameters
    ----------
    path : Directory path to create.

    Returns
    -------
    The same *path* object (for chaining).
    """
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_json(path: Path, payload: Dict[str, Any]) -> None:
    """Serialise *payload* as pretty-printed JSON and write to *path*.

    Creates any missing parent directories automatically.

    Parameters
    ----------
    path    : Destination file path (will be created or overwritten).
    payload : JSON-serialisable dict.
    """
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True)


def safe_float(value: Any) -> float:
    """Safely convert *value* to float, returning NaN on failure.

    Parameters
    ----------
    value : Any value to convert.

    Returns
    -------
    float representation of *value*, or ``float("nan")`` if conversion fails.
    """
    if value is None:
        return float("nan")
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")

## 3 · Data Contract Validator

**Gate function** — raises `ValueError` with a clear message if the processed dataset directory  
does not satisfy all structural requirements before any model code runs.

Checks enforced:
- All 5 required files present
- Channel order exactly `(CP1, CP2, CPz)`
- Metadata contains `trial_id` and `alignment` columns
- EEG array shape consistent with metadata row count

Writes `validation_report.json` on success.

In [10]:
def _resolve_required_columns(metadata: pd.DataFrame, config: DataContractConfig) -> Tuple[pd.DataFrame, List[str]]:
    renamed = metadata.copy()
    missing: List[str] = []
    for required in config.required_metadata_columns:
        if required in renamed.columns:
            continue
        aliases = config.optional_aliases.get(required, ())
        alias_match = next((alias for alias in aliases if alias in renamed.columns), None)
        if alias_match is None:
            if required == "alignment":
                renamed[required] = "response_locked"
                continue
            missing.append(required)
            continue
        renamed = renamed.rename(columns={alias_match: required})
    return renamed, missing


def _read_channel_names(path: Path) -> List[str]:
    with path.open("r", encoding="utf-8") as handle:
        return [line.strip() for line in handle.readlines() if line.strip()]


def _extract_sampling_rate(times_ms: np.ndarray) -> float:
    if len(times_ms) < 2:
        return float("nan")
    step_ms = float(np.mean(np.diff(times_ms)))
    if step_ms == 0:
        return float("nan")
    return 1000.0 / step_ms


def validate_stage2_dataset(dataset_dir: Path, output_dir: Path, config: DataContractConfig | None = None) -> Dict[str, object]:
    config = config or DataContractConfig()
    output_dir = ensure_dir(output_dir)

    file_checks = {name: (dataset_dir / name).exists() for name in config.expected_files}
    missing_files = [name for name, exists in file_checks.items() if not exists]

    report: Dict[str, object] = {
        "dataset_dir": str(dataset_dir),
        "contract": asdict(config),
        "file_checks": file_checks,
        "missing_files": missing_files,
        "passed": False,
    }

    if missing_files:
        write_json(output_dir / "stage1_blocking_issue_report.json", report)
        return report

    eeg = np.load(dataset_dir / "eeg_cpp_trials.npy")
    times_ms = np.load(dataset_dir / "times_ms.npy")
    metadata = pd.read_csv(dataset_dir / "metadata.csv")
    metadata, missing_columns = _resolve_required_columns(metadata, config)
    channels = _read_channel_names(dataset_dir / "channel_names.txt")
    notes_text = (dataset_dir / "preprocessing_notes.md").read_text(encoding="utf-8")

    n_trials, n_timepoints, n_channels = eeg.shape
    metadata_rows_match = len(metadata) == n_trials
    times_match = len(times_ms) == n_timepoints
    channel_order_matches = tuple(channels) == config.expected_channel_order
    sampling_rate_hz = _extract_sampling_rate(times_ms)

    required_columns_present = not missing_columns
    viable_mask = np.ones(len(metadata), dtype=bool)
    per_subject_summary: List[Dict[str, object]] = []

    report.update(
        {
            "shape_summary": {
                "n_trials": int(n_trials),
                "n_timepoints": int(n_timepoints),
                "n_channels": int(n_channels),
            },
            "metadata_rows_match": metadata_rows_match,
            "times_match": times_match,
            "channel_order_matches": channel_order_matches,
            "missing_metadata_columns": missing_columns,
            "sampling_rate_hz": sampling_rate_hz,
            "required_columns_present": required_columns_present,
            "per_subject_retained_trials": per_subject_summary,
            "preprocessing_policy_extract": {
                "reference_mentioned": "reference" in notes_text.lower(),
                "filter_mentioned": "filter" in notes_text.lower(),
                "artifact_mentioned": "artifact" in notes_text.lower() or "ica" in notes_text.lower(),
                "baseline_mentioned": "baseline" in notes_text.lower(),
            },
        }
    )

    report["passed"] = all(
        [
            metadata_rows_match,
            times_match,
            channel_order_matches,
            not missing_columns,
        ]
    )

    filename = "stage1_data_contract_report.json" if report["passed"] else "stage1_blocking_issue_report.json"
    write_json(output_dir / filename, report)
    if per_subject_summary:
        pd.DataFrame(per_subject_summary).to_csv(output_dir / "stage1_subject_trial_summary.csv", index=False)
    return report

## 4 · Dataset Pipeline

Loads 7 297 response-locked trials, z-normalises them, builds causal prediction targets and a valid-step mask, then wraps everything in PyTorch `DataLoader`s.

**Key design decisions:**

| Decision | Why |
|----------|-----|
| Z-normalise with **training-split stats only** | Prevents data leakage into val/test splits |
| Causal prediction target at *t* = EEG at *t+1* | Forces GRU to learn forward temporal dynamics |
| Mask excludes final `horizon_steps` points | Those time steps have no complete future-prediction target |
| Time weights: late window (`−120→−50 ms`) × 2.5 | Focuses the loss on the CPP build-up region |

#### `dataset.py` — § 1  Split Metadata Container

In [11]:
@dataclass
class Stage2SplitArtifacts:
    """Holds the trial indices and normalisation statistics for a data split.

    Attributes
    ----------
    train_indices : 1-D array of trial indices assigned to the training set.
    val_indices   : 1-D array of trial indices assigned to the validation set.
    test_indices  : 1-D array of trial indices assigned to the test set.
    train_mean    : (C,) per-channel mean computed on the training set only.
    train_std     : (C,) per-channel std  computed on the training set only.
    horizon_steps : Number of time steps in the causal prediction horizon.
    """

    train_indices: np.ndarray
    val_indices: np.ndarray
    test_indices: np.ndarray
    train_mean: np.ndarray
    train_std: np.ndarray
    horizon_steps: int


def _coerce_model_config(value: Any) -> ModelConfig:
    """Accept either a ModelConfig or a plain dict and return ModelConfig."""
    if isinstance(value, ModelConfig):
        return value
    if isinstance(value, dict):
        return ModelConfig(**value)
    raise TypeError(f"Unsupported model config type: {type(value)!r}")


def _coerce_loss_weights(value: Any) -> LossWeights:
    """Accept either LossWeights or a plain dict and return LossWeights."""
    if isinstance(value, LossWeights):
        return value
    if isinstance(value, dict):
        return LossWeights(**value)
    raise TypeError(f"Unsupported loss config type: {type(value)!r}")


def _coerce_training_config(config: Any) -> TrainingConfig:
    """Normalise legacy dict-style configs into a TrainingConfig instance."""
    if isinstance(config, TrainingConfig):
        return config
    if isinstance(config, dict):
        raw = dict(config)
    elif hasattr(config, "__dict__"):
        raw = dict(vars(config))
    else:
        raise TypeError(f"Unsupported training config type: {type(config)!r}")

    default_cfg = TrainingConfig()
    training_field_names = set(TrainingConfig.__dataclass_fields__.keys())
    model_field_names = set(ModelConfig.__dataclass_fields__.keys())
    loss_field_names = set(LossWeights.__dataclass_fields__.keys())

    model_source = raw.pop("model", default_cfg.model)
    loss_source = raw.pop("loss", default_cfg.loss)
    model_kwargs = dict(vars(_coerce_model_config(model_source)))
    loss_kwargs = dict(vars(_coerce_loss_weights(loss_source)))

    for key in list(raw.keys()):
        if key in model_field_names:
            model_kwargs[key] = raw.pop(key)
        elif key in loss_field_names:
            loss_kwargs[key] = raw.pop(key)

    clean_training_kwargs = {
        key: value for key, value in raw.items() if key in training_field_names and key not in {"model", "loss"}
    }
    clean_training_kwargs["model"] = ModelConfig(**model_kwargs)
    clean_training_kwargs["loss"] = LossWeights(**loss_kwargs)
    return TrainingConfig(**clean_training_kwargs)


def _extract_checkpoint_state_dict(ckpt: Dict[str, Any]) -> Dict[str, torch.Tensor]:
    """Support both current and legacy checkpoint key names."""
    for key in ("model_state_dict", "model_state"):
        if key in ckpt:
            return ckpt[key]
    raise KeyError("Checkpoint is missing model weights.")


def _translate_legacy_state_dict(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    """Rename legacy model parameter keys to the current model naming scheme."""
    prefix_map = {
        "input_projection.": "input_proj.",
        "layer_norm.": "input_norm.",
        "encoder.": "gru.",
        "reconstruction_head.net.": "recon_head.",
        "future_head.net.": "pred_head.",
    }
    translated: Dict[str, torch.Tensor] = {}
    for key, value in state_dict.items():
        new_key = key
        for old_prefix, new_prefix in prefix_map.items():
            if key.startswith(old_prefix):
                new_key = new_prefix + key[len(old_prefix):]
                break
        translated[new_key] = value
    return translated


def _load_checkpoint_weights(model: nn.Module, ckpt: Dict[str, Any]) -> None:
    """Load whichever checkpoint weights still match the current model shape."""
    raw_state = _extract_checkpoint_state_dict(ckpt)
    translated_state = _translate_legacy_state_dict(raw_state)
    current_state = model.state_dict()
    compatible_state = {
        key: value
        for key, value in translated_state.items()
        if key in current_state and tuple(current_state[key].shape) == tuple(value.shape)
    }
    model.load_state_dict(compatible_state, strict=False)

#### `dataset.py` — § 2  Mask Construction

In [12]:
def _build_trial_mask(
    times_ms: np.ndarray,
    n_trials: int,
    config: TrainingConfig,
    horizon_steps: int,
) -> np.ndarray:
    """Build a (n_trials, T) boolean mask for valid loss-computation time steps.

    A time step is considered valid when:
      1. It falls within ``config.analysis_window_ms`` (the pre-response EEG
         window of interest), AND
      2. It is not among the last ``horizon_steps`` steps of the epoch.
         Those final steps lack complete causal future targets and must be
         excluded to avoid training on zero-padded targets.

    Parameters
    ----------
    times_ms      : (T,) time axis in milliseconds (response-locked).
    n_trials      : Number of trials (first axis of the mask).
    config        : TrainingConfig carrying ``analysis_window_ms``.
    horizon_steps : Number of future steps in the prediction target; the last
                    ``horizon_steps`` time steps are excluded.

    Returns
    -------
    mask : (n_trials, T) bool array.
    """
    in_window = (
        (times_ms >= config.analysis_window_ms[0])
        & (times_ms <= config.analysis_window_ms[1])
    )  # (T,)

    # Exclude the last horizon_steps time steps (no complete future targets).
    has_future = np.arange(len(times_ms)) <= (len(times_ms) - horizon_steps - 1)  # (T,)

    row_mask = in_window & has_future  # (T,)
    return np.broadcast_to(row_mask[None, :], (n_trials, len(times_ms))).copy()


def build_pre_response_mask(
    times_ms: np.ndarray,
    window_end_ms: np.ndarray | float,
    min_mask_lead_ms: int,
) -> np.ndarray:
    """Build a variable-endpoint pre-response validity mask.

    Used in analyses where each trial has a different response time and we
    want to mask out post-response contamination per trial.

    Parameters
    ----------
    times_ms        : (T,) shared time axis.
    window_end_ms   : Scalar or (N,) array of per-trial window end times.
    min_mask_lead_ms: Safety margin in ms subtracted from each end time.

    Returns
    -------
    mask : (1, T) or (N, T) bool array.
    """
    times_ms = np.asarray(times_ms, dtype=np.float32)
    if np.isscalar(window_end_ms):
        threshold = (
            np.asarray([float(window_end_ms)], dtype=np.float32)
            - float(min_mask_lead_ms)
        )
    else:
        threshold = np.asarray(window_end_ms, dtype=np.float32) - float(min_mask_lead_ms)
    return (times_ms[None, :] >= 0.0) & (times_ms[None, :] <= threshold[:, None])

#### `dataset.py` — § 3  Split & Normalisation Helpers

In [13]:
def _random_trial_split(
    n_trials: int,
    config: TrainingConfig,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Randomly partition trials into train / val / test sets.

    Parameters
    ----------
    n_trials : Total number of trials.
    config   : TrainingConfig carrying seed and fraction fields.

    Returns
    -------
    train_indices, val_indices, test_indices : non-overlapping index arrays.
    """
    config = _coerce_training_config(config)
    indices = np.arange(n_trials)
    rng = np.random.default_rng(config.seed)
    rng.shuffle(indices)
    n_train = max(1, int(round(n_trials * config.train_fraction)))
    n_val   = max(1, int(round(n_trials * config.val_fraction)))
    n_train = min(n_train, max(1, n_trials - 2))
    n_val   = min(n_val,   max(1, n_trials - n_train - 1))
    train_indices = indices[:n_train]
    val_indices   = indices[n_train : n_train + n_val]
    test_indices  = indices[n_train + n_val :]
    if len(test_indices) == 0:
        test_indices = val_indices[-1:]
        val_indices  = val_indices[:-1]
    return train_indices, val_indices, test_indices


def _compute_channel_stats(
    eeg: np.ndarray,
    indices: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute per-channel mean and std from the training subset only.

    Parameters
    ----------
    eeg     : (N, T, C) raw EEG array.
    indices : Trial indices belonging to the training set.

    Returns
    -------
    mean : (C,) float32 array.
    std  : (C,) float32 array (zero channels replaced with 1.0).
    """
    train_data = eeg[indices]
    mean = train_data.mean(axis=(0, 1))
    std  = train_data.std(axis=(0, 1))
    std  = np.where(std == 0.0, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def _normalize_with_stats(
    eeg: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
) -> np.ndarray:
    """Apply z-score normalisation using pre-computed mean and std.

    Parameters
    ----------
    eeg  : (N, T, C) array to normalise.
    mean : (C,) per-channel mean.
    std  : (C,) per-channel std.

    Returns
    -------
    Normalised (N, T, C) float32 array.
    """
    return ((eeg - mean[None, None, :]) / std[None, None, :]).astype(np.float32)

#### `dataset.py` — § 4  Future-Target & Time-Weight Construction

In [14]:
def _build_future_targets(
    eeg: np.ndarray,
    horizon_steps: int,
) -> np.ndarray:
    """Build single-step-ahead causal prediction targets.

    For each time step t, the target is the EEG at time t+1
    (i.e. a 1-step horizon).  The last ``horizon_steps`` positions are
    left as zero because no complete future window exists there.

    Parameters
    ----------
    eeg           : (N, T, C) normalised EEG array.
    horizon_steps : Number of look-ahead steps (typically 1 at 1-step horizon).

    Returns
    -------
    targets : (N, T, C) float32 array — EEG at t+1 for each valid t.
    """
    n_trials, n_timepoints, n_channels = eeg.shape
    targets = np.zeros((n_trials, n_timepoints, n_channels), dtype=np.float32)
    # Shift EEG forward by 1 step: target[t] = eeg[t+1]
    if n_timepoints > 1:
        targets[:, :-1, :] = eeg[:, 1:, :]
    return targets


def _build_time_weights(
    times_ms: np.ndarray,
    config: TrainingConfig,
) -> np.ndarray:
    """Assign per-time-step training weights that up-weight the late pre-response window.

    Weights increase from early (1.0) → mid (1.75) → late (2.5) to
    steer the model toward the CPP build-up region that matters most
    for behaviour.

    Parameters
    ----------
    times_ms : (T,) time axis.
    config   : TrainingConfig carrying early/mid/late window boundaries.

    Returns
    -------
    weights : (T,) float32 array.
    """
    weights = np.zeros_like(times_ms, dtype=np.float32)
    early_mask = (times_ms >= config.early_window_ms[0]) & (times_ms < config.early_window_ms[1])
    mid_mask   = (times_ms >= config.mid_window_ms[0])   & (times_ms < config.mid_window_ms[1])
    late_mask  = (times_ms >= config.late_window_ms[0])  & (times_ms <= config.late_window_ms[1])
    weights[early_mask] = 1.0
    weights[mid_mask]   = 1.75
    weights[late_mask]  = 2.5
    return weights

#### `dataset.py` — § 5  PyTorch Dataset

In [15]:
class EEGWindowDataset(Dataset):
    """PyTorch Dataset wrapping a trial-level EEG array and its auxiliary tensors.

    Each item is a dict with keys:
      ``eeg``          : (T, C) float tensor — normalised EEG input.
      ``target_future``: (T, C) float tensor — one-step-ahead prediction target.
      ``mask``         : (T,)   float tensor — 1 at valid loss-computation steps.
      ``times_ms``     : (T,)   float tensor — shared time axis.
      ``trial_idx``    : ()     long tensor  — original trial index in the full dataset.

    Parameters
    ----------
    eeg      : (N, T, C) normalised EEG array.
    targets  : (N, T, C) future-target array.
    mask     : (N, T)    boolean/float mask array.
    times_ms : (T,)      time axis.
    indices  : 1-D array of trial indices to include in this split.
    """

    def __init__(
        self,
        eeg: np.ndarray,
        targets: np.ndarray,
        mask: np.ndarray,
        times_ms: np.ndarray,
        indices: np.ndarray,
    ) -> None:
        self.eeg      = torch.as_tensor(eeg[indices],     dtype=torch.float32)
        self.targets  = torch.as_tensor(targets[indices], dtype=torch.float32)
        self.mask     = torch.as_tensor(mask[indices],    dtype=torch.float32)
        self.times_ms = torch.as_tensor(times_ms,         dtype=torch.float32)
        self.indices  = torch.as_tensor(indices,          dtype=torch.long)

    def __len__(self) -> int:
        return int(self.eeg.shape[0])

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        return {
            "eeg":          self.eeg[idx],
            "target_future": self.targets[idx],
            "mask":         self.mask[idx],
            "times_ms":     self.times_ms,
            "trial_idx":    self.indices[idx],
        }

#### `dataset.py` — § 6  Public API

In [16]:
def load_stage2_dataset(
    dataset_dir: Path,
    config: TrainingConfig,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """Load, validate, normalise, and mask the Stage 2 EEG dataset.

    Parameters
    ----------
    dataset_dir : Directory containing ``eeg_cpp_trials.npy``, ``times_ms.npy``,
                  ``metadata.csv``, ``channel_names.txt``, and
                  ``preprocessing_notes.md`` (as required by DataContractConfig).
    config      : TrainingConfig driving split fractions, normalisation, and
                  mask boundaries.

    Returns
    -------
    eeg      : (N, T, C) float32 — z-scored EEG (training stats).
    targets  : (N, T, C) float32 — one-step-ahead prediction targets.
    mask     : (N, T)    bool    — valid time-step mask (analysis window,
                                   horizon boundary excluded).
    times_ms : (T,)      float32 — shared time axis.
    metadata : pd.DataFrame      — per-trial behavioural metadata.
    """
    config = _coerce_training_config(config)

    # --- Load raw arrays ------------------------------------------------------
    eeg = np.load(dataset_dir / "eeg_cpp_trials.npy").astype(np.float32)
    if not np.isfinite(eeg).all():
        eeg = np.nan_to_num(eeg, nan=0.0, posinf=0.0, neginf=0.0)
    times_ms = np.load(dataset_dir / "times_ms.npy").astype(np.float32)
    metadata = pd.read_csv(dataset_dir / "metadata.csv")

    # --- Contract validation --------------------------------------------------
    metadata, missing_columns = _resolve_required_columns(metadata, DataContractConfig())
    if missing_columns:
        raise ValueError(
            f"Missing required metadata columns after alias resolution: {missing_columns}"
        )
    channels = _read_channel_names(dataset_dir / "channel_names.txt")
    if tuple(channels) != DataContractConfig().expected_channel_order:
        raise ValueError(f"Unexpected channel order: {channels}")

    # --- Split & normalise ----------------------------------------------------
    train_idx, _, _ = _random_trial_split(len(metadata), config)
    train_mean, train_std = _compute_channel_stats(eeg, train_idx)
    eeg_norm = _normalize_with_stats(eeg, train_mean, train_std)

    # --- Causal horizon & targets ---------------------------------------------
    fs = 1000.0 / float(np.mean(np.diff(times_ms)))
    horizon_steps = max(1, int(round(config.future_horizon_ms * fs / 1000.0)))
    targets = _build_future_targets(eeg_norm, horizon_steps)

    # --- Mask (single shared call) -------------------------------------------
    n_trials = eeg_norm.shape[0]
    mask = _build_trial_mask(times_ms, n_trials, config, horizon_steps)

    return eeg_norm, targets, mask, times_ms, metadata


def make_dataloaders(
    eeg: np.ndarray,
    targets: np.ndarray,
    mask: np.ndarray,
    times_ms: np.ndarray,
    config: TrainingConfig,
) -> Tuple[DataLoader, DataLoader, DataLoader, Dict[str, np.ndarray]]:
    """Wrap normalised EEG arrays into train / val / test DataLoaders.

    Parameters
    ----------
    eeg      : (N, T, C) normalised EEG (output of load_stage2_dataset).
    targets  : (N, T, C) future-prediction targets.
    mask     : (N, T)    valid-step boolean mask.
    times_ms : (T,)      time axis.
    config   : TrainingConfig driving batch size, seed, and time weights.

    Returns
    -------
    train_loader : DataLoader shuffled over training trials.
    val_loader   : DataLoader over validation trials (no shuffle).
    test_loader  : DataLoader over test trials (no shuffle).
    split_indices: Dict with keys ``"train"``, ``"val"``, ``"test"`` mapping
                   to 1-D index arrays.
    """
    config = _coerce_training_config(config)
    set_global_seed(config.seed)
    train_idx, val_idx, test_idx = _random_trial_split(eeg.shape[0], config)
    time_weights = _build_time_weights(times_ms, config)

    # Scale mask by per-time-step weights (upweights the late CPP window).
    weighted_mask = mask * time_weights[None, :]  # (N, T) float

    def _make_loader(indices: np.ndarray, shuffle: bool) -> DataLoader:
        ds = EEGWindowDataset(eeg, targets, weighted_mask, times_ms, indices)
        return DataLoader(ds, batch_size=config.batch_size, shuffle=shuffle)

    train_loader = _make_loader(train_idx, shuffle=True)
    val_loader   = _make_loader(val_idx,   shuffle=False)
    test_loader  = _make_loader(test_idx,  shuffle=False)

    split_indices: Dict[str, np.ndarray] = {
        "train": train_idx,
        "val":   val_idx,
        "test":  test_idx,
    }
    return train_loader, val_loader, test_loader, split_indices

## 5 · Model Architecture — CPPForwardGRU

A **causal GRU encoder** — at every time step *t* the model sees only EEG up to and including *t*; no future information leaks into the hidden state.

```
Input (batch, T, 3)
  → Linear projection  (3 → 16) + LayerNorm
  → Causal GRU         (hidden_dim=32, num_layers=1)
  → Reconstruction head    (hidden → 3)        # predict current EEG frame
  → Future-prediction head (hidden → 3 × H)   # predict next H=50 ms of EEG
```

**Output:** `ForwardOutputs(reconstruction, future_prediction, latents)`  
— `latents` has shape `(batch, T, 32)` and is the representation used in all downstream analyses.

The **composite self-supervised loss** (`masked_self_supervised_loss`) has 11 terms  
all controlled via `LossWeights`.  **No behavioural labels are used during training.**

#### `model.py` — § 1  Output Container

In [17]:
@dataclass
class ForwardOutputs:
    """Named container for all outputs produced by a single CPPForwardGRU forward pass.

    Attributes
    ----------
    reconstructed : (B, T, C) tensor — per-channel EEG reconstruction at each time step.
    predicted     : (B, T, C) tensor — one-step-ahead causal prediction at each time step.
    latents       : (B, T, H) tensor — GRU hidden states (the learned latent representation).
    """

    reconstructed: torch.Tensor
    predicted: torch.Tensor
    latents: torch.Tensor

#### `model.py` — § 2  Model Definition

In [18]:
class CPPForwardGRU(nn.Module):
    """Causal GRU encoder for CPP-related EEG latent dynamics.

    Architecture (left-to-right)
    ----------------------------
    Input  (B, T, C)
      └─ Linear projection  C → projection_dim
      └─ LayerNorm
      └─ GRU (causal, num_layers stacked)   hidden = hidden_dim
      └─ Reconstruction head  hidden_dim → C   (what the model saw)
      └─ Prediction head      hidden_dim → C   (what comes next)

    Parameters
    ----------
    n_channels   : Number of EEG channels in the input (typically 3: CP1, CP2, CPz).
    model_config : :class:`ModelConfig` instance carrying projection_dim, hidden_dim,
                   and num_layers.  Pass ``cfg.model`` from a :class:`TrainingConfig`.
    """

    def __init__(self, n_channels: int, model_config: ModelConfig) -> None:
        super().__init__()
        self.n_channels = n_channels
        self.cfg = model_config

        # --- Input projection -------------------------------------------------
        self.input_proj = nn.Linear(n_channels, model_config.projection_dim)
        self.input_norm = nn.LayerNorm(model_config.projection_dim)

        # --- Causal recurrent encoder ----------------------------------------
        # batch_first=True keeps the (B, T, *) convention throughout.
        self.gru = nn.GRU(
            input_size=model_config.projection_dim,
            hidden_size=model_config.hidden_dim,
            num_layers=model_config.num_layers,
            batch_first=True,
        )

        # --- Output heads -----------------------------------------------------
        self.recon_head = nn.Sequential(
            nn.Linear(model_config.hidden_dim, model_config.hidden_dim),
            nn.ReLU(),
            nn.Linear(model_config.hidden_dim, n_channels),
        )
        self.pred_head = nn.Sequential(
            nn.Linear(model_config.hidden_dim, model_config.hidden_dim),
            nn.ReLU(),
            nn.Linear(model_config.hidden_dim, n_channels),
        )

    def forward(self, x: torch.Tensor) -> ForwardOutputs:
        """Run the causal forward pass.

        Parameters
        ----------
        x : (B, T, C) float tensor — normalised EEG input.

        Returns
        -------
        ForwardOutputs
            reconstructed : (B, T, C)
            predicted     : (B, T, C)
            latents       : (B, T, H)
        """
        projected = self.input_norm(self.input_proj(x))   # (B, T, proj_dim)
        hidden, _ = self.gru(projected)                    # (B, T, H)
        reconstructed = self.recon_head(hidden)            # (B, T, C)
        predicted = self.pred_head(hidden)                 # (B, T, C)
        return ForwardOutputs(
            reconstructed=reconstructed,
            predicted=predicted,
            latents=hidden,
        )

#### `model.py` — § 3  Loss Sub-functions

In [19]:
def _compute_reconstruction_losses(
    outputs: ForwardOutputs,
    target_current: torch.Tensor,
    target_future: torch.Tensor,
    mask: torch.Tensor,
    weights: LossWeights,
) -> Dict[str, torch.Tensor]:
    """Compute reconstruction, future-prediction, derivative, and variance losses.

    Parameters
    ----------
    outputs        : ForwardOutputs from the model forward pass.
    target_current : (B, T, C) — EEG signal the model should reconstruct.
    target_future  : (B, T, C) — one-step-ahead EEG targets.
    mask           : (B, T)    — float/bool mask (1 = valid time step).
    weights        : LossWeights controlling lambda values.

    Returns
    -------
    Dict mapping loss-name → scalar tensor.
    """
    mask_f = mask.float()
    n_valid = mask_f.sum().clamp(min=1.0)

    # --- 1a. Reconstruction loss (MSE, masked) --------------------------------
    recon_err = ((outputs.reconstructed - target_current) ** 2).mean(dim=-1)
    recon_loss = (recon_err * mask_f).sum() / n_valid

    # --- 1b. Future-prediction loss (MSE, down-weighted near window edges) ---
    pred_mask = mask_f * weights.future_weight_scale
    pred_err = ((outputs.predicted - target_future) ** 2).mean(dim=-1)
    future_loss = (pred_err * pred_mask).sum() / pred_mask.sum().clamp(min=1.0)

    # --- 1c. Derivative matching loss ----------------------------------------
    # Encourages the reconstructed waveform to have the same local slope as target.
    if target_current.shape[1] > 1:
        recon_diff = outputs.reconstructed[:, 1:, :] - outputs.reconstructed[:, :-1, :]
        target_diff = target_current[:, 1:, :] - target_current[:, :-1, :]
        deriv_err = ((recon_diff - target_diff) ** 2).mean(dim=-1)
        deriv_mask = mask_f[:, 1:]
        derivative_loss = (deriv_err * deriv_mask).sum() / deriv_mask.sum().clamp(min=1.0)
    else:
        derivative_loss = recon_loss.new_zeros(())

    # --- 1d. Variance alignment loss -----------------------------------------
    # Prevents the model from learning a flat mean and ignoring trial variance.
    recon_var = outputs.reconstructed.var(dim=1).mean()
    target_var = target_current.var(dim=1).mean()
    variance_loss = (recon_var - target_var).abs()

    return {
        "recon_loss": recon_loss,
        "future_loss": future_loss,
        "derivative_loss": derivative_loss,
        "variance_loss": variance_loss,
    }


def _compute_cpp_shape_prior_losses(
    outputs: ForwardOutputs,
    target_current: torch.Tensor,
    mask: torch.Tensor,
    times_ms: torch.Tensor,
    weights: LossWeights,
) -> Dict[str, torch.Tensor]:
    """Compute the CPP shape-prior sub-group losses.

    These losses encode soft inductive biases about the CPP waveform shape:
    monotonic build-up, non-zero slope, and late-window amplitude.
    All four terms are scaled by ``weights.lambda_cpp_prior`` in addition to
    their individual lambdas, so setting that to 0.0 disables the whole group.

    The CPP proxy is defined as the mean across the three CPP channels (CP1/CP2/CPz),
    which approximates the classic CPP scoring used in the literature.

    Parameters
    ----------
    outputs        : ForwardOutputs from the model forward pass.
    target_current : (B, T, C) — EEG signal used to compute target CPP proxy.
    mask           : (B, T)    — float/bool mask (1 = valid time step).
    times_ms       : (T,)      — time axis in milliseconds (response-locked).
    weights        : LossWeights carrying shape-prior lambdas and time windows.

    Returns
    -------
    Dict mapping loss-name → scalar tensor (all zero when prior is disabled).
    """
    zero = target_current.new_zeros(())

    if not weights.enable_cpp_shape_prior:
        return {
            "cpp_mean_loss": zero,
            "monotonic_loss": zero,
            "slope_floor_loss": zero,
            "late_amplitude_loss": zero,
            "cpp_mean_alignment_loss": zero,
        }

    # CPP proxy = mean across channels (shape B, T)
    recon_cpp: torch.Tensor = outputs.reconstructed.mean(dim=-1)
    target_cpp: torch.Tensor = target_current.mean(dim=-1)

    # Analysis-window 1-D mask
    analysis_mask_1d = (
        (times_ms >= weights.analysis_window_ms[0])
        & (times_ms <= weights.analysis_window_ms[1])
    )  # (T,)

    # Late-window 1-D mask
    late_mask_1d = (
        (times_ms >= weights.late_window_ms[0])
        & (times_ms <= weights.late_window_ms[1])
    )  # (T,)

    mask_f = mask.float()  # (B, T)
    analysis_mask_2d = analysis_mask_1d.float().unsqueeze(0) * mask_f  # (B, T)

    # --- 3a. CPP mean MSE loss ------------------------------------------------
    cpp_mean_err = (recon_cpp - target_cpp) ** 2
    n_ana = analysis_mask_2d.sum().clamp(min=1.0)
    cpp_mean_loss = (cpp_mean_err * analysis_mask_2d).sum() / n_ana

    # --- 3b. Monotonicity loss ------------------------------------------------
    # Penalise downward steps in the reconstructed CPP within the analysis window.
    if recon_cpp.shape[1] > 1:
        steps = recon_cpp[:, 1:] - recon_cpp[:, :-1]  # (B, T-1)
        ana_step_mask = analysis_mask_2d[:, 1:]
        monotonic_loss = (torch.clamp(-steps, min=0.0) * ana_step_mask).sum() / ana_step_mask.sum().clamp(min=1.0)
    else:
        monotonic_loss = zero

    # --- 3c. Slope floor loss -------------------------------------------------
    # Penalise the recon slope being below a fraction of the target slope.
    if target_cpp.shape[1] > 1:
        recon_slope = recon_cpp[:, 1:] - recon_cpp[:, :-1]
        target_slope = target_cpp[:, 1:] - target_cpp[:, :-1]
        slope_floor = weights.slope_floor_ratio * target_slope
        ana_step_mask = analysis_mask_2d[:, 1:]
        slope_deficit = torch.clamp(slope_floor - recon_slope, min=0.0)
        slope_floor_loss = (slope_deficit * ana_step_mask).sum() / ana_step_mask.sum().clamp(min=1.0)
    else:
        slope_floor_loss = zero

    # --- 3d. Late amplitude loss ----------------------------------------------
    # Penalise under-shooting the late CPP amplitude (last 70 ms before response).
    late_mask_2d = late_mask_1d.float().unsqueeze(0) * mask_f  # (B, T)
    n_late = late_mask_2d.sum().clamp(min=1.0)
    recon_late_mean = (recon_cpp * late_mask_2d).sum(dim=1) / late_mask_2d.sum(dim=1).clamp(min=1.0)
    target_late_mean = (target_cpp * late_mask_2d).sum(dim=1) / late_mask_2d.sum(dim=1).clamp(min=1.0)
    late_amplitude_loss = torch.clamp(target_late_mean - recon_late_mean, min=0.0).mean()

    # --- 3e. CPP mean alignment loss -----------------------------------------
    # Coarser global alignment across the full analysis window.
    recon_ana_mean = (recon_cpp * analysis_mask_2d).sum(dim=1) / analysis_mask_2d.sum(dim=1).clamp(min=1.0)
    target_ana_mean = (target_cpp * analysis_mask_2d).sum(dim=1) / analysis_mask_2d.sum(dim=1).clamp(min=1.0)
    cpp_mean_alignment_loss = ((recon_ana_mean - target_ana_mean) ** 2).mean()

    return {
        "cpp_mean_loss": cpp_mean_loss,
        "monotonic_loss": monotonic_loss,
        "slope_floor_loss": slope_floor_loss,
        "late_amplitude_loss": late_amplitude_loss,
        "cpp_mean_alignment_loss": cpp_mean_alignment_loss,
    }


def _compute_smoothness_loss(outputs: ForwardOutputs) -> torch.Tensor:
    """Penalise large frame-to-frame jumps in the GRU latent state.

    A smooth latent trajectory is a weak prior that encourages the model to
    learn slowly-varying dynamics rather than frame-by-frame noise fitting.

    Parameters
    ----------
    outputs : ForwardOutputs — latents shape (B, T, H).

    Returns
    -------
    Scalar tensor (0.0 when T <= 1).
    """
    if outputs.latents.shape[1] <= 1:
        return outputs.latents.new_zeros(())
    delta = outputs.latents[:, 1:, :] - outputs.latents[:, :-1, :]  # (B, T-1, H)
    return (delta ** 2).mean()

#### `model.py` — § 4  Composite Loss

In [20]:
def masked_self_supervised_loss(
    outputs: ForwardOutputs,
    target_current: torch.Tensor,
    target_future: torch.Tensor,
    mask: torch.Tensor,
    times_ms: torch.Tensor,
    weights: LossWeights,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Compute the full composite self-supervised loss and return a metrics dict.

    This function combines four loss groups into a single differentiable scalar:
      1. Reconstruction & prediction  (recon, future, derivative, variance)
      2. CPP proxy alignment          (cpp_mean)
      3. CPP shape prior              (monotonic, slope_floor, late_amplitude,
                                       cpp_mean_alignment) — can be disabled
      4. Latent smoothness            (smooth)

    Parameters
    ----------
    outputs        : ForwardOutputs produced by CPPForwardGRU.forward().
    target_current : (B, T, C) — EEG signal to reconstruct.
    target_future  : (B, T, C) — one-step-ahead EEG prediction targets.
    mask           : (B, T) bool/float — 1 at valid (non-horizon) time steps.
    times_ms       : (T,) — response-locked time axis in milliseconds.
    weights        : LossWeights carrying all lambda values and toggle flags.

    Returns
    -------
    total_loss : scalar tensor (differentiable).
    metrics    : Dict[str, float] with individual loss values for logging.
                 Always contains ``"total_loss"``.  Shape-prior terms are
                 present but zero when ``weights.enable_cpp_shape_prior`` is
                 ``False``.
    """
    # --- Group 1: Reconstruction & prediction --------------------------------
    recon_losses = _compute_reconstruction_losses(
        outputs, target_current, target_future, mask, weights
    )

    # --- Group 2: CPP proxy mean loss ----------------------------------------
    # (Simpler than shape prior; always active.)
    cpp_proxy = outputs.reconstructed.mean(dim=-1)          # (B, T)
    target_proxy = target_current.mean(dim=-1)               # (B, T)
    mask_f = mask.float()
    n_valid = mask_f.sum().clamp(min=1.0)
    cpp_mean_loss = (((cpp_proxy - target_proxy) ** 2) * mask_f).sum() / n_valid

    # --- Group 3: CPP shape prior --------------------------------------------
    prior_losses = _compute_cpp_shape_prior_losses(
        outputs, target_current, mask, times_ms, weights
    )

    # --- Group 4: Latent smoothness ------------------------------------------
    smooth_loss = _compute_smoothness_loss(outputs)

    # --- Weighted sum --------------------------------------------------------
    total_loss = (
        weights.lambda_recon       * recon_losses["recon_loss"]
        + weights.lambda_future    * recon_losses["future_loss"]
        + weights.lambda_derivative * recon_losses["derivative_loss"]
        + weights.lambda_variance  * recon_losses["variance_loss"]
        + weights.lambda_cpp_mean  * cpp_mean_loss
        + weights.lambda_cpp_prior * (
            weights.lambda_monotonic          * prior_losses["monotonic_loss"]
            + weights.lambda_slope_floor      * prior_losses["slope_floor_loss"]
            + weights.lambda_late_amplitude   * prior_losses["late_amplitude_loss"]
            + weights.lambda_cpp_mean_alignment * prior_losses["cpp_mean_alignment_loss"]
        )
        + weights.lambda_smooth    * smooth_loss
    )

    metrics: Dict[str, float] = {
        "total_loss":              total_loss.item(),
        "recon_loss":              recon_losses["recon_loss"].item(),
        "future_loss":             recon_losses["future_loss"].item(),
        "derivative_loss":         recon_losses["derivative_loss"].item(),
        "variance_loss":           recon_losses["variance_loss"].item(),
        "cpp_mean_loss":           cpp_mean_loss.item(),
        "monotonic_loss":          prior_losses["monotonic_loss"].item(),
        "slope_floor_loss":        prior_losses["slope_floor_loss"].item(),
        "late_amplitude_loss":     prior_losses["late_amplitude_loss"].item(),
        "cpp_mean_alignment_loss": prior_losses["cpp_mean_alignment_loss"].item(),
        "smooth_loss":             smooth_loss.item(),
    }
    return total_loss, metrics

## 6 · Training Pipeline

**AdamW** optimiser with weight decay, gradient clipping (`clip=1.0`), per-epoch validation monitoring, and early stopping (patience = 15 epochs).  
The checkpoint with the lowest validation loss is saved to `best_model.pt`.

`export_full_latents_from_checkpoint()` runs the **frozen** best model in `eval()` mode  
over all 7 297 trials and saves the complete hidden-state tensor `(7297, 308, 32)`  
to `latents_full.npz`.  This file is the input to all downstream analyses.

#### `train.py` — § 1  Training Utilities

In [21]:
def _run_epoch(
    model: CPPForwardGRU,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None,
    config: TrainingConfig,
    device: torch.device,
    *,
    train: bool,
) -> Dict[str, float]:
    """Run one full epoch (training or evaluation) and return averaged metrics.

    Parameters
    ----------
    model     : CPPForwardGRU instance.
    loader    : DataLoader yielding batches from EEGWindowDataset.
    optimizer : Adam optimiser (None when ``train=False``).
    config    : TrainingConfig carrying gradient clip and loss weights.
    device    : Torch device to run on.
    train     : If True, compute gradients and update parameters.

    Returns
    -------
    Dict[str, float] with averaged loss metrics for the epoch.
    """
    model.train(train)
    accum: Dict[str, float] = {}
    n_batches = 0

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            x         = batch["eeg"].to(device)
            x_future  = batch["target_future"].to(device)
            mask      = batch["mask"].to(device)
            times_ms  = batch["times_ms"][0].to(device)  # shared across batch

            outputs: ForwardOutputs = model(x)
            loss, metrics = masked_self_supervised_loss(
                outputs, x, x_future, mask, times_ms, config.loss
            )

            if train and optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
                optimizer.step()

            for k, v in metrics.items():
                accum[k] = accum.get(k, 0.0) + v
            n_batches += 1

    if n_batches == 0:
        return accum
    return {k: v / n_batches for k, v in accum.items()}

#### `train.py` — § 2  Visualisation Helpers

In [22]:
def _save_loss_curves(
    train_losses: list[float],
    val_losses: list[float],
    output_dir: Path,
) -> None:
    """Save a training / validation loss curve plot to *output_dir*.

    Parameters
    ----------
    train_losses : Per-epoch training total losses.
    val_losses   : Per-epoch validation total losses.
    output_dir   : Directory to write ``loss_curve.png`` into.
    """
    try:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(train_losses, label="train")
        ax.plot(val_losses,   label="val")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Total loss")
        ax.set_title("Training curve")
        ax.legend()
        fig.tight_layout()
        fig.savefig(output_dir / "loss_curve.png", dpi=120)
        plt.close(fig)
    except Exception:
        pass  # Non-critical; skip silently if matplotlib is unavailable.


def _save_reconstruction_examples(
    model: CPPForwardGRU,
    loader: DataLoader,
    output_dir: Path,
    device: torch.device,
    n_examples: int = 8,
) -> None:
    """Save a grid of per-trial reconstruction vs. input waveforms.

    Parameters
    ----------
    model      : Trained CPPForwardGRU.
    loader     : DataLoader to sample trials from (first batch used).
    output_dir : Directory to write ``reconstruction_examples.png`` into.
    device     : Torch device.
    n_examples : Number of trials to display.
    """
    try:
        model.eval()
        batch = next(iter(loader))
        x = batch["eeg"][:n_examples].to(device)
        times_ms = batch["times_ms"][0].cpu().numpy()
        with torch.no_grad():
            out = model(x)
        x_np   = x.cpu().numpy()
        rec_np = out.reconstructed.cpu().numpy()

        fig, axes = plt.subplots(n_examples, 1, figsize=(10, 2 * n_examples), sharex=True)
        for i, ax in enumerate(axes):
            ax.plot(times_ms, x_np[i, :, :].mean(axis=-1), label="input",  lw=1.2)
            ax.plot(times_ms, rec_np[i, :, :].mean(axis=-1), label="recon", lw=1.2, ls="--")
            if i == 0:
                ax.legend(fontsize=8)
        axes[-1].set_xlabel("Time (ms)")
        fig.suptitle("Reconstruction examples (CPP proxy)")
        fig.tight_layout()
        fig.savefig(output_dir / "reconstruction_examples.png", dpi=120)
        plt.close(fig)
    except Exception:
        pass


def _save_cpp_average_plot(
    model: CPPForwardGRU,
    loader: DataLoader,
    output_dir: Path,
    device: torch.device,
) -> None:
    """Compute and save the grand-average CPP waveform comparison.

    Saves both a PNG figure and a ``.npz`` file (``cpp_average_sanity.npz``)
    with keys ``recon_mean``, ``target_mean``, and ``times_ms``.

    .. note::
        ``_save_cpp_comparison_overlay`` reads the ``.npz`` written here, so
        this function **must** run before that one.

    Parameters
    ----------
    model      : Trained CPPForwardGRU.
    loader     : DataLoader (full split, no shuffle preferred).
    output_dir : Directory for output files.
    device     : Torch device.
    """
    try:
        model.eval()
        all_recon, all_input, times_np = [], [], None
        with torch.no_grad():
            for batch in loader:
                x = batch["eeg"].to(device)
                times_ms = batch["times_ms"][0].cpu().numpy()
                out = model(x)
                all_recon.append(out.reconstructed.cpu().numpy().mean(axis=-1))
                all_input.append(x.cpu().numpy().mean(axis=-1))
                times_np = times_ms

        recon_mean  = np.concatenate(all_recon,  axis=0).mean(axis=0)
        target_mean = np.concatenate(all_input, axis=0).mean(axis=0)

        # Persist for downstream overlay plot.
        np.savez(output_dir / "cpp_average_sanity.npz",
                 recon_mean=recon_mean, target_mean=target_mean, times_ms=times_np)

        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(times_np, target_mean, label="target CPP", lw=1.5)
        ax.plot(times_np, recon_mean,  label="recon CPP",  lw=1.5, ls="--")
        ax.axvline(0, color="k", lw=0.8, ls=":")
        ax.set_xlabel("Time from response (ms)")
        ax.legend()
        ax.set_title("Grand-average CPP: target vs reconstruction")
        fig.tight_layout()
        fig.savefig(output_dir / "cpp_average_comparison.png", dpi=150)
        plt.close(fig)
    except Exception:
        pass


def _save_cpp_comparison_overlay(output_dir: Path) -> None:
    """Overlay derivative comparison on the grand-average CPP figure.

    Prerequisite: ``_save_cpp_average_plot`` must have written
    ``cpp_average_sanity.npz`` to *output_dir*.

    Parameters
    ----------
    output_dir : Directory containing ``cpp_average_sanity.npz``.
    """
    try:
        npz_path = output_dir / "cpp_average_sanity.npz"
        if not npz_path.exists():
            return
        data = np.load(npz_path)
        recon_mean  = data["recon_mean"]
        target_mean = data["target_mean"]
        times_np    = data["times_ms"]

        recon_deriv  = np.gradient(recon_mean,  times_np)
        target_deriv = np.gradient(target_mean, times_np)

        fig, axes = plt.subplots(1, 2, figsize=(12, 3))
        axes[0].plot(times_np, target_mean, label="target"); axes[0].plot(times_np, recon_mean, ls="--", label="recon")
        axes[0].set_title("CPP amplitude"); axes[0].legend()
        axes[1].plot(times_np, target_deriv, label="target slope"); axes[1].plot(times_np, recon_deriv, ls="--", label="recon slope")
        axes[1].set_title("CPP slope (d/dt)"); axes[1].legend()
        for ax in axes:
            ax.axvline(0, color="k", lw=0.8, ls=":")
            ax.set_xlabel("Time (ms)")
        fig.tight_layout()
        fig.savefig(output_dir / "cpp_derivative_comparison.png", dpi=150)
        plt.close(fig)
    except Exception:
        pass

#### `train.py` — § 3  Main Training Loop

In [23]:
def train_model(
    dataset_dir: Path,
    output_dir: Path,
    config: TrainingConfig,
) -> Dict[str, object]:
    """Train CPPForwardGRU with early stopping and save the best checkpoint.

    Parameters
    ----------
    dataset_dir : Directory containing the processed EEG dataset
                  (output of the preprocessing pipeline).
    output_dir  : Directory to write the checkpoint, loss curve, and
                  diagnostic figures into.
    config      : Full TrainingConfig (architecture + training + loss weights).

    Returns
    -------
    Dict with keys:
      ``"best_val_loss"``    : float — best validation total loss achieved.
      ``"checkpoint_path"``  : Path  — absolute path to ``best_model.pt``.
      ``"n_epochs_trained"`` : int   — total epochs run (including patience).
      ``"train_losses"``     : list[float].
      ``"val_losses"``       : list[float].
    """
    config = _coerce_training_config(config)
    set_global_seed(config.seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- Data ----------------------------------------------------------------
    eeg, targets, mask, times_ms, metadata = load_stage2_dataset(dataset_dir, config)
    train_loader, val_loader, test_loader, split_indices = make_dataloaders(
        eeg, targets, mask, times_ms, config
    )
    n_channels = eeg.shape[-1]

    # --- Model & optimiser ---------------------------------------------------
    model = CPPForwardGRU(n_channels=n_channels, model_config=config.model).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )

    # --- Training loop -------------------------------------------------------
    best_val_loss = float("inf")
    patience_counter = 0
    train_losses: list[float] = []
    val_losses:   list[float] = []
    checkpoint_path = output_dir / "best_model.pt"

    for epoch in range(config.max_epochs):
        train_metrics = _run_epoch(model, train_loader, optimizer, config, device, train=True)
        val_metrics   = _run_epoch(model, val_loader,   None,      config, device, train=False)

        train_losses.append(train_metrics["total_loss"])
        val_losses.append(val_metrics["total_loss"])

        if val_metrics["total_loss"] < best_val_loss:
            best_val_loss = val_metrics["total_loss"]
            patience_counter = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": config,
                    "epoch": epoch,
                    "val_loss": best_val_loss,
                    "split_indices": split_indices,
                },
                checkpoint_path,
            )
        else:
            patience_counter += 1
            if patience_counter >= config.early_stopping_patience:
                break

    # --- Diagnostic figures --------------------------------------------------
    _save_loss_curves(train_losses, val_losses, output_dir)
    # Reload best weights for figure generation.
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    _load_checkpoint_weights(model, ckpt)
    _save_reconstruction_examples(model, val_loader, output_dir, device)
    _save_cpp_average_plot(model, val_loader, output_dir, device)
    _save_cpp_comparison_overlay(output_dir)

    return {
        "best_val_loss":   best_val_loss,
        "checkpoint_path": checkpoint_path,
        "n_epochs_trained": len(train_losses),
        "train_losses": train_losses,
        "val_losses":   val_losses,
    }

#### `train.py` — § 4  Latent Export

In [24]:
def export_full_latents_from_checkpoint(
    checkpoint_path: Path,
    dataset_dir: Path,
    output_dir: Path,
) -> Path:
    """Export full-dataset GRU hidden states from a saved checkpoint.

    Runs the model in eval mode over *all* trials (train + val + test) and
    saves the resulting latent tensor to a ``.npz`` file.

    Parameters
    ----------
    checkpoint_path : Path to ``best_model.pt`` (written by :func:`train_model`).
    dataset_dir     : Dataset directory (same as used during training).
    output_dir      : Directory to write ``latents_full.npz`` into.

    Returns
    -------
    Path to the saved ``latents_full.npz`` file.

    Output ``.npz`` keys
    --------------------
    ``"latents"``  : (N, T, H) float32 — hidden states for all N trials.
    ``"times_ms"`` : (T,)      float32 — shared response-locked time axis.
    """
    checkpoint_path = Path(checkpoint_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- Load checkpoint & config --------------------------------------------
    ckpt   = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = _coerce_training_config(ckpt["config"])

    # --- Reload dataset (all trials, no split filtering) --------------------
    eeg, targets, mask, times_ms, metadata = load_stage2_dataset(dataset_dir, config)
    n_channels = eeg.shape[-1]

    # --- Rebuild model & load weights ----------------------------------------
    model = CPPForwardGRU(n_channels=n_channels, model_config=config.model).to(device)
    _load_checkpoint_weights(model, ckpt)
    model.eval()

    # --- Inference in mini-batches (avoids OOM on large datasets) ------------
    batch_size = 256
    all_latents: list[np.ndarray] = []
    eeg_tensor = torch.as_tensor(eeg, dtype=torch.float32)

    with torch.no_grad():
        for start in range(0, len(eeg), batch_size):
            x_batch = eeg_tensor[start : start + batch_size].to(device)
            out     = model(x_batch)
            all_latents.append(out.latents.cpu().numpy())

    latents_full = np.concatenate(all_latents, axis=0)  # (N, T, H)
    out_path = output_dir / "latents_full.npz"
    np.savez(out_path, latents=latents_full, times_ms=times_ms)

    return out_path

### Inline Visual Preview Helpers

These helpers show key figures directly inside the notebook after each major stage.

In [ ]:
def _display_saved_figure(path: Path, title: Optional[str] = None) -> None:
    """Display a saved image inline when it exists."""
    path = Path(path)
    if title:
        print(title)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Figure not found: {path}")


def _preview_latent_states(latent_npz: Path, n_dims: int = 4, n_trials: int = 6) -> None:
    """Show a quick inline preview of the exported latent states."""
    latent_npz = Path(latent_npz)
    if not latent_npz.exists():
        print(f"Latent file not found: {latent_npz}")
        return

    data = np.load(latent_npz)
    latents = data["latents"].astype(np.float32)
    times_ms = data["times_ms"].astype(np.float32)

    n_dims = max(1, min(n_dims, latents.shape[-1]))
    n_trials = max(1, min(n_trials, latents.shape[0]))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    mean_latents = latents[:, :, :n_dims].mean(axis=0)
    for dim_idx in range(n_dims):
        axes[0].plot(times_ms, mean_latents[:, dim_idx], label=f"dim {dim_idx}")
    axes[0].axvline(0, color="k", lw=0.8, ls=":")
    axes[0].set_title("Average latent trajectories")
    axes[0].set_xlabel("Time (ms)")
    axes[0].set_ylabel("Latent value")
    axes[0].legend(loc="best")

    preview = latents[:n_trials, :, 0].reshape(n_trials, len(times_ms))
    im = axes[1].imshow(preview, aspect="auto", cmap="coolwarm", origin="lower")
    axes[1].set_title("Latent dim 0 across sample trials")
    axes[1].set_xlabel("Time index")
    axes[1].set_ylabel("Trial")
    xticks = np.linspace(0, len(times_ms) - 1, num=min(6, len(times_ms)), dtype=int)
    axes[1].set_xticks(xticks)
    axes[1].set_xticklabels([f"{times_ms[i]:.0f}" for i in xticks])
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    fig.tight_layout()
    plt.show()


def _show_checkpoint_diagnostics(
    checkpoint_path: Path,
    dataset_dir: Path,
    n_examples: int = 6,
) -> None:
    """Render key model diagnostics inline from an existing checkpoint."""
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        print(f"Checkpoint not found: {checkpoint_path}")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = _coerce_training_config(ckpt["config"])
    eeg, targets, mask, times_ms, metadata = load_stage2_dataset(dataset_dir, config)
    _, val_loader, _, _ = make_dataloaders(eeg, targets, mask, times_ms, config)

    model = CPPForwardGRU(n_channels=eeg.shape[-1], model_config=config.model).to(device)
    _load_checkpoint_weights(model, ckpt)
    model.eval()

    loss_curve_path = checkpoint_path.parent / "loss_curve.png"
    if loss_curve_path.exists():
        _display_saved_figure(loss_curve_path, "Training curve")
    else:
        print("Training curve")
        print("This checkpoint does not include a saved loss-curve image. Re-training will recreate it.")

    batch = next(iter(val_loader))
    x = batch["eeg"][:n_examples].to(device)
    preview_times = batch["times_ms"][0].cpu().numpy()
    with torch.no_grad():
        out = model(x)
    x_np = x.cpu().numpy()
    rec_np = out.reconstructed.cpu().numpy()

    fig, axes = plt.subplots(n_examples, 1, figsize=(10, 2 * n_examples), sharex=True)
    if n_examples == 1:
        axes = [axes]
    for idx, ax in enumerate(axes):
        ax.plot(preview_times, x_np[idx, :, :].mean(axis=-1), label="input", lw=1.2)
        ax.plot(preview_times, rec_np[idx, :, :].mean(axis=-1), label="recon", lw=1.2, ls="--")
        ax.axvline(0, color="k", lw=0.8, ls=":")
        if idx == 0:
            ax.legend(fontsize=8)
    axes[-1].set_xlabel("Time (ms)")
    fig.suptitle("Reconstruction examples")
    fig.tight_layout()
    plt.show()

    all_recon, all_input = [], []
    with torch.no_grad():
        for batch in val_loader:
            x_batch = batch["eeg"].to(device)
            out_batch = model(x_batch)
            all_recon.append(out_batch.reconstructed.cpu().numpy().mean(axis=-1))
            all_input.append(x_batch.cpu().numpy().mean(axis=-1))
    recon_mean = np.concatenate(all_recon, axis=0).mean(axis=0)
    target_mean = np.concatenate(all_input, axis=0).mean(axis=0)
    recon_deriv = np.gradient(recon_mean, preview_times)
    target_deriv = np.gradient(target_mean, preview_times)

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].plot(preview_times, target_mean, label="target", lw=1.5)
    axes[0].plot(preview_times, recon_mean, label="recon", lw=1.5, ls="--")
    axes[0].axvline(0, color="k", lw=0.8, ls=":")
    axes[0].set_title("Grand-average CPP")
    axes[0].set_xlabel("Time (ms)")
    axes[0].legend()
    axes[1].plot(preview_times, target_deriv, label="target slope", lw=1.5)
    axes[1].plot(preview_times, recon_deriv, label="recon slope", lw=1.5, ls="--")
    axes[1].axvline(0, color="k", lw=0.8, ls=":")
    axes[1].set_title("CPP slope comparison")
    axes[1].set_xlabel("Time (ms)")
    axes[1].legend()
    fig.tight_layout()
    plt.show()


## 7 · ▶ Run Training

Trains `CPPForwardGRU` with the composite self-supervised loss.  
**Skip this cell** if `best_model.pt` already exists.

> ⏱ ~10–20 min on CPU.  Max 100 epochs; early stopping patience = 15.


In [25]:
_cfg = TrainingConfig(
    seed=42,
    model=ModelConfig(projection_dim=16, hidden_dim=32, num_layers=1),
    loss=LossWeights(
        lambda_recon=1.0,
        lambda_future=0.2,
        lambda_cpp_prior=0.1,
        enable_cpp_shape_prior=True,
    ),
)

if CHECKPOINT_PATH.exists():
    print(f"Checkpoint already exists at: {CHECKPOINT_PATH}")
    print("Delete it and re-run this cell to train from scratch.")
else:
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    train_model(
        dataset_dir=DATASET_DIR,
        output_dir=RESULTS_DIR / "model_checkpoints",
        config=_cfg,
    )
    print("Training complete.")

_show_checkpoint_diagnostics(CHECKPOINT_PATH, DATASET_DIR)

Checkpoint already exists at: /Users/siyu/Documents/GitHub/biiigProject/Results/model_checkpoints/best_model.pt
Delete it and re-run this cell to train from scratch.


## 8 · ▶ Export Full Latent States

Run the **frozen** best model in `eval()` mode over all 7 297 trials.

Output: `Data/IntermediateData/latents_full/latents_full.npz`
- `latents`  : shape `(7297, 308, 32)` — hidden state at every time point
- `times_ms` : shape `(308,)`
- `trial_ids`: shape `(7297,)`, row-aligned with `metadata.csv`

This file is the **input to all downstream analyses** (S3 audit, ridge regression, future CPP latent-axis analysis).


In [30]:
if LATENT_PATH.exists():
    _z = np.load(LATENT_PATH)
    print(f"Latents already exported — shape: {_z['latents'].shape}")
else:
    INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
    export_full_latents_from_checkpoint(
        checkpoint_path=CHECKPOINT_PATH,
        dataset_dir=DATASET_DIR,
        output_dir=INTERMEDIATE_DIR,
    )
    print("Latents exported successfully.")

_preview_latent_states(LATENT_PATH)

AttributeError: 'dict' object has no attribute 'seed'

## 9 · Ridge Regression — RT Analysis

Tests whether **time-averaged GRU hidden states** can predict `log(RT_ms)` beyond subject-level baselines.

**Method:**
- 4 pre-response averaging windows: `−600→−300`, `−300→−120`, `−120→−50`, `−600→−50` ms
- Nested cross-validation: outer 5-fold subject-stratified KFold + inner held-out validation for α selection
- 4 model types compared: baseline · hidden-only · baseline+hidden · baseline+shuffled-hidden

**Key result (logged in `logs.md`):**  
Early window `−600→−300 ms`:  baseline R²=0.197  →  baseline+hidden R²=0.300  (Δ R²=**+0.103**)  
Shuffled-hidden control R²=0.195 confirms the increment is genuine signal, not overfitting.

#### `rt_ridge.py` — § 1  Data Loading & Input Validation

In [ ]:
def _load_latents(latent_npz: Path) -> Tuple[np.ndarray, np.ndarray]:
    """Load latent hidden states and the shared time axis from a .npz file.

    Parameters
    ----------
    latent_npz : Path to ``latents_full.npz`` written by
                 :func:`export_full_latents_from_checkpoint`.

    Returns
    -------
    latents  : (N, T, H) float32 array of GRU hidden states.
    times_ms : (T,) float32 array of response-locked time values.

    Raises
    ------
    KeyError  : If required keys ``"latents"`` or ``"times_ms"`` are missing.
    ValueError: If array shapes are inconsistent.
    """
    data = np.load(latent_npz)
    if "latents" not in data or "times_ms" not in data:
        raise KeyError(f"latents_full.npz must contain 'latents' and 'times_ms'; got {list(data.keys())}")
    latents  = data["latents"].astype(np.float32)   # (N, T, H)
    times_ms = data["times_ms"].astype(np.float32)  # (T,)
    if latents.ndim != 3:
        raise ValueError(f"Expected latents shape (N, T, H), got {latents.shape}")
    if times_ms.ndim != 1 or times_ms.shape[0] != latents.shape[1]:
        raise ValueError(
            f"times_ms length {times_ms.shape[0]} does not match latents T={latents.shape[1]}"
        )
    return latents, times_ms


def _load_behaviour(dataset_dir: Path) -> pd.DataFrame:
    """Load and minimally validate the trial-level behavioural metadata.

    Parameters
    ----------
    dataset_dir : Directory containing ``metadata.csv``.

    Returns
    -------
    pd.DataFrame with at least columns ``RT_ms`` and ``subject_id``.

    Raises
    ------
    FileNotFoundError : If ``metadata.csv`` is absent.
    KeyError          : If required columns are missing.
    """
    csv_path = Path(dataset_dir) / "metadata.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"metadata.csv not found in {dataset_dir}")
    df = pd.read_csv(csv_path)
    required = {"RT_ms", "subject_id"}
    missing  = required - set(df.columns)
    if missing:
        raise KeyError(f"metadata.csv is missing required columns: {missing}")
    return df

#### `rt_ridge.py` — § 2  Feature Engineering

In [ ]:
# --- 2a. Time-window averaging of hidden states ----------------------------

def _window_features(
    latents: np.ndarray,
    times_ms: np.ndarray,
    window_ms: Tuple[float, float],
) -> np.ndarray:
    """Average hidden states within a pre-response time window.

    Parameters
    ----------
    latents   : (N, T, H) GRU hidden-state array.
    times_ms  : (T,) response-locked time axis.
    window_ms : (start_ms, end_ms) inclusive boundaries.

    Returns
    -------
    (N, H) float32 array — mean hidden state over the window.

    Raises
    ------
    ValueError : If the window contains no valid time steps.
    """
    mask = (times_ms >= window_ms[0]) & (times_ms <= window_ms[1])
    if not mask.any():
        raise ValueError(
            f"Window {window_ms} ms contains no time steps. "
            f"times_ms ranges from {times_ms.min():.1f} to {times_ms.max():.1f} ms."
        )
    return latents[:, mask, :].mean(axis=1).astype(np.float32)  # (N, H)


# --- 2b. Baseline design matrix -------------------------------------------

def _baseline_design(df: pd.DataFrame) -> np.ndarray:
    """Build a per-trial baseline feature matrix from behavioural metadata.

    Baseline features
    -----------------
    - One-hot subject dummies (mean-centred; first subject dropped to avoid
      multicollinearity).
    - Normalised difficulty (z-scored ``coherence`` or ``difficulty`` column
      when present; otherwise a column of zeros).
    - Accuracy flag (``correctness`` column as 0/1 when present).

    Parameters
    ----------
    df : Trial-level metadata DataFrame.

    Returns
    -------
    (N, K) float32 design matrix.
    """
    parts: List[np.ndarray] = []

    # Subject dummies (drop first to avoid perfect multicollinearity).
    subj_dummies = pd.get_dummies(df["subject_id"], drop_first=True).values.astype(np.float32)
    parts.append(subj_dummies)

    # Difficulty / coherence (optional).
    diff_col = next((c for c in ("coherence", "difficulty") if c in df.columns), None)
    if diff_col is not None:
        vals = df[diff_col].values.astype(np.float32)
        std  = vals.std()
        parts.append(((vals - vals.mean()) / (std if std > 0 else 1.0)).reshape(-1, 1))
    else:
        parts.append(np.zeros((len(df), 1), dtype=np.float32))

    # Accuracy (optional).
    if "correctness" in df.columns:
        parts.append(df["correctness"].values.astype(np.float32).reshape(-1, 1))

    return np.concatenate(parts, axis=1)  # (N, K)


# --- 2c. Hand-crafted CPP features (for external validation) ---------------

def _cpp_features(dataset_dir: Path, df: pd.DataFrame) -> Optional[np.ndarray]:
    """Extract hand-crafted CPP amplitude and slope features when available.

    These are used as an external-validation baseline to test whether hidden
    states provide information *beyond* directly measured CPP features.

    Parameters
    ----------
    dataset_dir : Dataset directory (may contain ``eeg_cpp_trials.npy``).
    df          : Metadata DataFrame (used for alignment checks).

    Returns
    -------
    (N, 2) float32 array with columns [late_amplitude, pre_response_slope],
    or ``None`` if the EEG file is not available.
    """
    eeg_path = Path(dataset_dir) / "eeg_cpp_trials.npy"
    if not eeg_path.exists():
        return None
    eeg = np.load(eeg_path).astype(np.float32)  # (N, T, C)
    if eeg.shape[0] != len(df):
        return None  # Alignment mismatch — skip rather than corrupt.

    times_ms = np.load(Path(dataset_dir) / "times_ms.npy").astype(np.float32)
    cpp = eeg.mean(axis=-1)  # (N, T) — mean across CP1/CP2/CPz

    # Late amplitude: mean CPP in [-120, -50] ms.
    late_mask = (times_ms >= -120.0) & (times_ms <= -50.0)
    late_amp  = cpp[:, late_mask].mean(axis=1) if late_mask.any() else np.zeros(len(df))

    # Pre-response slope: linear regression coefficient in [-300, -50] ms.
    slope_mask = (times_ms >= -300.0) & (times_ms <= -50.0)
    if slope_mask.sum() >= 2:
        t_slope = times_ms[slope_mask]
        t_norm  = (t_slope - t_slope.mean()) / t_slope.std()
        slopes  = np.array([np.polyfit(t_norm, cpp[i, slope_mask], 1)[0] for i in range(len(df))])
    else:
        slopes = np.zeros(len(df))

    return np.stack([late_amp, slopes], axis=1).astype(np.float32)


# =============================================================================
# § 3  Ridge Regression & Cross-Validation
#      Outer KFold (grouped by subject) + inner alpha selection
# =============================================================================

_ALPHA_GRID = np.logspace(-3, 4, 30)


def _fit_predict_outer_cv(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    n_outer_folds: int = 5,
    scale: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """Leave-group-out Ridge regression with inner alpha tuning.

    Uses grouped K-Fold (grouped by subject ID) for the outer loop so that
    test subjects never appear in the training fold.  Alpha is chosen by
    inner 5-fold CV on the training fold only.

    Parameters
    ----------
    X             : (N, K) design matrix (baseline or baseline + hidden).
    y             : (N,)   target variable (log RT).
    groups        : (N,)   subject group labels for outer fold assignment.
    n_outer_folds : Number of outer CV folds (default: 5).
    scale         : Whether to z-score X within each fold (default: True).

    Returns
    -------
    y_pred  : (N,) out-of-fold predictions.
    y_true  : (N,) corresponding true values (same order as X rows).
    """
    kf     = KFold(n_splits=n_outer_folds, shuffle=True, random_state=42)
    y_pred = np.empty_like(y)

    for train_idx, test_idx in kf.split(X, y, groups=groups):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te       = X[test_idx]

        if scale:
            scaler = StandardScaler()
            X_tr   = scaler.fit_transform(X_tr)
            X_te   = scaler.transform(X_te)

        ridge = RidgeCV(alphas=_ALPHA_GRID, cv=5)
        ridge.fit(X_tr, y_tr)
        y_pred[test_idx] = ridge.predict(X_te)

    return y_pred, y

#### `rt_ridge.py` — § 4  Summary Statistics & Delta Metrics

In [ ]:
def _r2_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute coefficient of determination R²."""
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")


def _summarise_performance(
    results: Dict[str, Dict[str, float]]
) -> pd.DataFrame:
    """Convert a nested results dict into a tidy summary DataFrame.

    Parameters
    ----------
    results : {window_label: {model_label: r2_value}} mapping.

    Returns
    -------
    pd.DataFrame with columns ``window``, ``model``, ``r2``.
    """
    rows = [
        {"window": window, "model": model, "r2": r2}
        for window, models in results.items()
        for model, r2 in models.items()
    ]
    return pd.DataFrame(rows)

#### `rt_ridge.py` — § 5  Visualisation & Output

In [ ]:
def _save_performance_figure(
    df_perf: pd.DataFrame,
    output_dir: Path,
) -> None:
    """Save a grouped bar chart of R² by time window and model.

    Parameters
    ----------
    df_perf    : Tidy DataFrame with columns ``window``, ``model``, ``r2``.
    output_dir : Directory to write ``ridge_rt_performance.png`` into.
    """
    try:
        windows = df_perf["window"].unique()
        models  = df_perf["model"].unique()
        x       = np.arange(len(windows))
        width   = 0.8 / max(len(models), 1)

        fig, ax = plt.subplots(figsize=(8, 4))
        for i, model in enumerate(models):
            vals = [
                df_perf.loc[(df_perf["window"] == w) & (df_perf["model"] == model), "r2"].values[0]
                if ((df_perf["window"] == w) & (df_perf["model"] == model)).any()
                else float("nan")
                for w in windows
            ]
            ax.bar(x + i * width, vals, width, label=model)

        ax.set_xticks(x + width * (len(models) - 1) / 2)
        ax.set_xticklabels(windows, rotation=20, ha="right")
        ax.set_ylabel("R²  (out-of-fold)")
        ax.set_title("Ridge RT prediction: R² by time window and model")
        ax.legend()
        fig.tight_layout()
        fig.savefig(output_dir / "ridge_rt_performance.png", dpi=150)
        plt.close(fig)
    except Exception:
        pass  # Non-critical visualisation.


def _save_delta_figure(
    df_delta: pd.DataFrame,
    output_dir: Path,
) -> None:
    """Save a bar chart of ΔR² (hidden states vs. baseline) per window.

    Parameters
    ----------
    df_delta   : DataFrame with columns ``window`` and ``delta_r2``.
    output_dir : Directory to write ``ridge_rt_deltas.png`` into.
    """
    try:
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.bar(df_delta["window"], df_delta["delta_r2"])
        ax.axhline(0, color="k", lw=0.8)
        ax.set_ylabel("ΔR²  (baseline+hidden − baseline)")
        ax.set_title("Incremental RT predictability from hidden states")
        ax.tick_params(axis="x", rotation=20)
        fig.tight_layout()
        fig.savefig(output_dir / "ridge_rt_deltas.png", dpi=150)
        plt.close(fig)
    except Exception:
        pass

#### `rt_ridge.py` — § 6  Public Entry Point

In [ ]:
def run_ridge_rt_analysis(
    latent_npz: Path,
    dataset_dir: Path,
    output_dir: Path,
    window_definitions: Optional[Dict[str, Tuple[float, float]]] = None,
    n_outer_folds: int = 5,
) -> Dict[str, Any]:
    """Run Ridge regression predicting log(RT) from GRU hidden states.

    For each time window, two nested-CV Ridge models are evaluated:
      * ``"baseline"``         — subject dummies + difficulty + accuracy.
      * ``"baseline+hidden"``  — baseline features + window-averaged hidden states.

    Optionally (when CPP EEG data is present):
      * ``"baseline+cpp"``     — baseline + hand-crafted CPP features.
      * ``"baseline+cpp+hidden"`` — all three.

    Parameters
    ----------
    latent_npz         : Path to ``latents_full.npz``.
    dataset_dir        : Dataset directory (for ``metadata.csv`` and optionally
                         ``eeg_cpp_trials.npy``).
    output_dir         : Directory for CSV and figure outputs.
    window_definitions : Dict mapping window label → (start_ms, end_ms).
                         Defaults to four canonical pre-response windows.
    n_outer_folds      : Number of outer CV folds (default: 5).

    Returns
    -------
    Dict with key ``"performance"`` — a nested dict
    ``{window_label: {model_label: r2_value}}`` — and key ``"delta_r2"``
    — a dict ``{window_label: delta_r2_value}`` for the hidden-state increment.

    Output files
    ------------
    ``ridge_rt_performance.csv`` : Tidy R² table.
    ``ridge_rt_deltas.csv``      : ΔR² (hidden increment) per window.
    ``ridge_rt_performance.png`` : Grouped bar chart.
    ``ridge_rt_deltas.png``      : Delta bar chart.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if window_definitions is None:
        window_definitions = {
            "early  (−600 to −300 ms)": (-600.0, -300.0),
            "mid    (−300 to −120 ms)": (-300.0, -120.0),
            "late   (−120 to  −50 ms)": (-120.0,  -50.0),
            "full   (−600 to  −50 ms)": (-600.0,  -50.0),
        }

    # --- Load inputs ----------------------------------------------------------
    latents,  times_ms = _load_latents(Path(latent_npz))
    df_beh             = _load_behaviour(Path(dataset_dir))
    n_trials           = latents.shape[0]

    if len(df_beh) != n_trials:
        raise ValueError(
            f"latents has {n_trials} trials but metadata has {len(df_beh)} rows. "
            "Ensure both were built from the same dataset."
        )

    # --- Target variable: log(RT) ------------------------------------------
    log_rt = np.log(df_beh["RT_ms"].values.astype(np.float64))

    # --- Baseline design matrix -------------------------------------------
    X_baseline = _baseline_design(df_beh)

    # --- Optional CPP features -------------------------------------------
    X_cpp = _cpp_features(Path(dataset_dir), df_beh)

    # --- Subject groups (for grouped K-Fold) ------------------------------
    groups = df_beh["subject_id"].values

    # --- Evaluate per window ----------------------------------------------
    performance: Dict[str, Dict[str, float]] = {}
    delta_r2:    Dict[str, float]             = {}

    for label, window_ms in window_definitions.items():
        try:
            X_hidden = _window_features(latents, times_ms, window_ms)
        except ValueError:
            continue  # Skip windows that fall outside the time axis.

        X_aug = np.concatenate([X_baseline, X_hidden], axis=1)

        preds_base,  y_true = _fit_predict_outer_cv(X_baseline, log_rt, groups, n_outer_folds)
        preds_aug,   _      = _fit_predict_outer_cv(X_aug,      log_rt, groups, n_outer_folds)

        r2_base = _r2_score(y_true, preds_base)
        r2_aug  = _r2_score(y_true, preds_aug)

        window_results: Dict[str, float] = {
            "baseline":        r2_base,
            "baseline+hidden": r2_aug,
        }

        if X_cpp is not None:
            X_cpp_aug  = np.concatenate([X_baseline, X_cpp], axis=1)
            X_all      = np.concatenate([X_baseline, X_cpp, X_hidden], axis=1)
            preds_cpp, _ = _fit_predict_outer_cv(X_cpp_aug, log_rt, groups, n_outer_folds)
            preds_all, _ = _fit_predict_outer_cv(X_all,     log_rt, groups, n_outer_folds)
            window_results["baseline+cpp"]        = _r2_score(y_true, preds_cpp)
            window_results["baseline+cpp+hidden"] = _r2_score(y_true, preds_all)

        performance[label] = window_results
        delta_r2[label]    = r2_aug - r2_base

    # --- Persist results --------------------------------------------------
    df_perf  = _summarise_performance(performance)
    df_delta = pd.DataFrame([
        {"window": w, "delta_r2": d} for w, d in delta_r2.items()
    ])

    df_perf.to_csv(output_dir  / "ridge_rt_performance.csv", index=False)
    df_delta.to_csv(output_dir / "ridge_rt_deltas.csv",      index=False)

    _save_performance_figure(df_perf,  output_dir)
    _save_delta_figure(df_delta, output_dir)

    return {"performance": performance, "delta_r2": delta_r2}

## 10 · ▶ Run Ridge Regression

Predict `log(RT_ms)` from window-averaged hidden states.  
All outputs are written to `Results/regression/`.


In [ ]:
_ridge_results = run_ridge_rt_analysis(
    latent_npz=LATENT_PATH,
    dataset_dir=DATASET_DIR,
    output_dir=RESULTS_DIR / "regression",
)
print("Ridge RT analysis complete.")
display(_summarise_performance(_ridge_results["performance"]))
display(pd.DataFrame([
    {"window": window, "delta_r2": delta}
    for window, delta in _ridge_results["delta_r2"].items()
]))
_display_saved_figure(RESULTS_DIR / "regression" / "ridge_rt_performance.png", "Ridge RT performance")
_display_saved_figure(RESULTS_DIR / "regression" / "ridge_rt_deltas.png", "Increment from hidden states")

# Display performance table if available
_tbl = RESULTS_DIR/"regression"/"ridge_rt_hidden_rt_rerun"/"performance_summary.csv"
if _tbl.exists():
    print()
    print(pd.read_csv(_tbl).to_string(index=False))

---
## 11 · Results Summary & Next Steps

### Pipeline outputs

| Stage | Output path |
|-------|------------|
| Data validation | `Results/validation/validation_report.json` |
| Best model checkpoint | `Results/model_checkpoints/best_model.pt` |
| Full latent states | `Data/IntermediateData/latents_full/latents_full.npz` (`7297 × 308 × 32`) |
| Ridge RT results | `Results/regression/ridge_rt_hidden_rt_rerun/` |
| Publication figures | `Results/figures/publication/` |

### Current scientific status

| Claim | Evidence | Status |
|-------|----------|--------|
| Model reconstructs CPP waveform | corr = 0.99, R² = 0.88 | ✅ Established |
| Hidden states encode CPP amplitude | Δ R² = 0.95 vs baseline | ✅ Established |
| Hidden states predict RT beyond subject baseline | Δ R² ≈ +0.103 (early window) | ⚠️  Descriptive only |
| Latents outperform hand-crafted CPP features for RT | Not yet tested | ❌ Pending |
| DDM drift-rate regression | Not yet done | ❌ Pending |

### Next steps (in priority order)

1. **Define a formal CPP latent axis**  
   Use PCA or linear regression on `latents_full.npz` to extract a single direction in hidden space  
   that maximally aligns with CPP amplitude.  Test cross-fold and cross-subject stability.  
   → `Scripts/s4_analysis/analysis.py`

2. **Test incremental RT prediction**  
   Compare: `baseline + CPP latent axis projection` vs `baseline + hand-crafted CPP (amplitude + slope)`.  
   If the latent axis wins, we have evidence that the learned representation adds value beyond direct measurement.  
   → modify `Scripts/s4_analysis/rt_ridge.py`

3. **DDM drift-rate regression**  
   Fit per-subject DDM, extract per-trial drift rate estimates, use as regression target.  
   This is the mechanistic claim: latent states encode evidence-accumulation rate.  
   → new script under `Scripts/s4_analysis/`
